# Lab: End-to-End MLP Pipeline — Face Recognition (Olivetti Faces)
### SOLUTION NOTEBOOK (Instructor Reference)

**Objective:** Build a complete deep learning pipeline using a Multilayer Perceptron (MLP)
in PyTorch — from raw data to a trained, evaluated classifier.

**We will:**
1. Load and explore the dataset
2. Preprocess the data (handle missing values, encode/split, scale)
3. Design an MLP and choose activation functions
4. Choose a loss function and optimizer
5. Train the model manually (batch-by-batch)
6. Evaluate and visualize results

**Dataset:** Olivetti Faces — 400 grayscale images (64×64) of 40 people (10 images each).
Task: predict *which person* is in the image (40-class classification).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
data = fetch_olivetti_faces()
X, y = data.data, data.target   # X: (400, 4096) flattened images, y: (400,) person labels 0-39

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of classes:", len(np.unique(y)))
print("Pixel value range:", X.min(), "to", X.max())

# ---- Visualize a few samples ----
fig, axes = plt.subplots(1, 5, figsize=(10, 3))
for i, ax in enumerate(axes):
    ax.imshow(X[i].reshape(64, 64), cmap='gray')
    ax.set_title(f"Person {y[i]}")
    ax.axis('off')
plt.suptitle("Sample Faces")
plt.show()


### Step 1: Preprocessing
Every ML pipeline needs these 3 things done properly:
1. Handle missing values (if any)
2. Split into train/test sets
3. Scale the features


In [ ]:
# 1. Check for missing values
has_missing = np.isnan(X).any()
print("Any missing values?", has_missing)
# Olivetti Faces has none, but we check generically for pipeline completeness
if has_missing:
    col_means = np.nanmean(X, axis=0)
    inds = np.where(np.isnan(X))
    X[inds] = np.take(col_means, inds[1])

# 2. Train/test split — stratified so every person appears in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3. Feature scaling — fit ONLY on train, then transform both
# (Fitting on test too would leak test-set statistics into training — "data leakage")
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


### Step 2: Convert data to PyTorch tensors
PyTorch models need `torch.Tensor` inputs, not NumPy arrays.
- Use `dtype=torch.float32` for features (X)
- Use `dtype=torch.long` for labels (y) — required for classification


In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
X_test_t  = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t  = torch.tensor(y_test, dtype=torch.long).to(device)

print("Train tensor shape:", X_train_t.shape)
print("Test tensor shape:", X_test_t.shape)


### Step 3: Design the MLP

Structure: `Input (4096) → Hidden Layer 1 → Activation → Hidden Layer 2 → Activation → Output (40)`

- Hidden layers: 512 and 128
- Activation: **ReLU** — avoids vanishing gradients, faster convergence than Sigmoid/Tanh in hidden layers
- No Dropout here — intentionally left out (not yet taught); this makes some train/test overfitting
  gap likely, which is a useful teaching setup for introducing Dropout in a future lab
- No Softmax on the output layer — raw logits are passed directly to `CrossEntropyLoss`, which applies
  softmax internally


In [ ]:
class FaceMLP(nn.Module):
    def __init__(self, input_dim=4096, num_classes=40):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),              # ReLU: avoids vanishing gradients, fast convergence in hidden layers
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)   # raw logits — softmax is applied inside CrossEntropyLoss
        )

    def forward(self, x):
        return self.net(x)

model = FaceMLP().to(device)
print(model)


### Step 4: Choose loss function and optimizer

- **Loss:** `nn.CrossEntropyLoss()` — combines log-softmax + negative log-likelihood in one step,
  the standard choice for multi-class classification.
- **Optimizer:** `Adam` — adapts the learning rate per parameter automatically, converges faster than
  plain SGD, and is a strong default for small MLPs like this one.


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)


### Step 5: Train the model

We haven't used `DataLoader` here — batching and shuffling are done manually using `torch.randperm`.

For each epoch:
1. Shuffle the training data indices
2. Loop through the data in chunks of `batch_size`
3. For each batch: zero gradients → forward pass → compute loss → backward pass → optimizer step
4. Track total loss and accuracy for the epoch


In [ ]:
num_epochs = 60
batch_size = 16
n_samples = X_train_t.size(0)

train_losses, train_accs = [], []

for epoch in range(num_epochs):
    model.train()

    perm = torch.randperm(n_samples)   # shuffle indices each epoch

    running_loss, correct, total = 0.0, 0, 0

    for i in range(0, n_samples, batch_size):
        idx = perm[i:i+batch_size]
        xb, yb = X_train_t[idx], y_train_t[idx]

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f} Train Acc: {epoch_acc:.4f}")


### Step 6: Evaluate on the test set

Set the model to eval mode, run a forward pass on the test set with no gradient tracking,
and compute test accuracy.


In [ ]:
model.eval()

with torch.no_grad():
    outputs = model(X_test_t)
    preds = outputs.argmax(dim=1)

all_preds = preds.cpu().numpy()
all_labels = y_test_t.cpu().numpy()

test_acc = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {test_acc:.4f}")

# ---- Loss curve ----
plt.figure(figsize=(6,4))
plt.plot(train_losses, label="Train Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training Loss Curve")
plt.legend(); plt.show()

# ---- Confusion matrix ----
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10,10))
plt.imshow(cm, cmap='Blues')
plt.title("Confusion Matrix (40 classes)")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.colorbar()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = np.random.randint(0, len(X_test))
    img = X_test[idx].reshape(64, 64)
    true_label = y_test[idx]
    with torch.no_grad():
        pred = model(torch.tensor(X_test[idx:idx+1], dtype=torch.float32).to(device)).argmax(dim=1).item()
    ax.imshow(img, cmap='gray')
    ax.set_title(f"True: {true_label} | Pred: {pred}", fontsize=9,
                 color='green' if pred == true_label else 'red')
    ax.axis('off')
plt.suptitle("Sample Predictions")
plt.tight_layout()
plt.show()
